In [1]:
import os
import cv2
import xml.etree.ElementTree as ET

# =========================
# CONFIG
# =========================
img_folder = "sampleimages/DJI_0004_HS0600D/images/train"
xml_folder = "annotations"
out_img = "chunked_images"
out_xml = "chunked_annotations"

chunk_width = 640
img_height_expected = 225   # your image height

os.makedirs(out_img, exist_ok=True)
os.makedirs(out_xml, exist_ok=True)

# =========================
# PROCESS
# =========================
for file in os.listdir(img_folder):
    if not file.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    img_path = os.path.join(img_folder, file)
    xml_path = os.path.join(xml_folder, os.path.splitext(file)[0] + ".xml")

    img = cv2.imread(img_path)
    if img is None:
        continue

    h, w = img.shape[:2]

    if h != img_height_expected:
        print(f"Skip {file}: height {h}")
        continue

    if not os.path.exists(xml_path):
        print(f"No XML for {file}")
        continue

    tree = ET.parse(xml_path)
    root = tree.getroot()

    # read all objects first
    objects = []
    for obj in root.findall("object"):
        name = obj.find("name").text

        box = obj.find("bndbox")
        xmin = int(box.find("xmin").text)
        ymin = int(box.find("ymin").text)
        xmax = int(box.find("xmax").text)
        ymax = int(box.find("ymax").text)

        objects.append((name, xmin, ymin, xmax, ymax))

    # make chunks
    num_chunks = (w + chunk_width - 1) // chunk_width

    for i in range(num_chunks):
        x1 = i * chunk_width
        x2 = min((i + 1) * chunk_width, w)

        chunk = img[:, x1:x2]

        base = os.path.splitext(file)[0]
        new_name = f"{base}_chunk{i}"

        # save image
        cv2.imwrite(os.path.join(out_img, new_name + ".jpg"), chunk)

        # create XML
        ann = ET.Element("annotation")

        ET.SubElement(ann, "folder").text = out_img
        ET.SubElement(ann, "filename").text = new_name + ".jpg"

        size = ET.SubElement(ann, "size")
        ET.SubElement(size, "width").text = str(x2 - x1)
        ET.SubElement(size, "height").text = str(h)
        ET.SubElement(size, "depth").text = "3"

        # add intersecting boxes
        for name, xmin, ymin, xmax, ymax in objects:

            # skip if no overlap
            if xmax <= x1 or xmin >= x2:
                continue

            # clip into chunk
            new_xmin = max(xmin, x1) - x1
            new_xmax = min(xmax, x2) - x1

            obj = ET.SubElement(ann, "object")
            ET.SubElement(obj, "name").text = name
            ET.SubElement(obj, "pose").text = "Unspecified"
            ET.SubElement(obj, "truncated").text = "0"
            ET.SubElement(obj, "difficult").text = "0"

            box = ET.SubElement(obj, "bndbox")
            ET.SubElement(box, "xmin").text = str(new_xmin)
            ET.SubElement(box, "ymin").text = str(ymin)
            ET.SubElement(box, "xmax").text = str(new_xmax)
            ET.SubElement(box, "ymax").text = str(ymax)

        tree_out = ET.ElementTree(ann)
        tree_out.write(os.path.join(out_xml, new_name + ".xml"))

print("Done.")

No XML for frame_000000.png
No XML for frame_002397.png
No XML for frame_004794.png
No XML for frame_007191.png
No XML for frame_009588.png
No XML for frame_011985.png
Done.


In [6]:
import os
import cv2

# ==========================================
# CONFIG
# ==========================================
base_path = r"sampleimages\DJI_0004_HS0600D"

splits = ["train", "val"]

chunk_width = 640
overlap = 0  # set >0 if wanted later

img_exts = [".jpg", ".jpeg", ".png"]

# output folders
out_base = base_path + "_chunked"

# ==========================================
# HELPERS
# ==========================================
def find_image_file(folder, stem):
    for ext in img_exts:
        p = os.path.join(folder, stem + ext)
        if os.path.exists(p):
            return p
    return None


def yolo_to_pixel(xc, yc, bw, bh, img_w, img_h):
    x1 = (xc - bw / 2) * img_w
    y1 = (yc - bh / 2) * img_h
    x2 = (xc + bw / 2) * img_w
    y2 = (yc + bh / 2) * img_h
    return x1, y1, x2, y2


def pixel_to_yolo(x1, y1, x2, y2, w, h):
    bw = x2 - x1
    bh = y2 - y1
    xc = x1 + bw / 2
    yc = y1 + bh / 2

    return (
        xc / w,
        yc / h,
        bw / w,
        bh / h
    )


# ==========================================
# MAIN
# ==========================================
for split in splits:

    img_dir = os.path.join(base_path, "images", split)
    lbl_dir = os.path.join(base_path, "labels", split)

    out_img_dir = os.path.join(out_base, "images", split)
    out_lbl_dir = os.path.join(out_base, "labels", split)

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    for file in os.listdir(lbl_dir):

        if not file.endswith(".txt"):
            continue

        stem = os.path.splitext(file)[0]
        # print(f"stem={stem}")

        img_path = find_image_file(img_dir, stem)
        print(img_dir + stem)
        if img_path is None:
            print("Image not found:", stem)
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]

        # read labels
        boxes = []
        with open(os.path.join(lbl_dir, file), "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue

                cls = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:])
                x1, y1, x2, y2 = yolo_to_pixel(xc, yc, bw, bh, w, h)

                boxes.append((cls, x1, y1, x2, y2))

        # chunk loop
        step = chunk_width - overlap
        chunk_id = 0

        for start_x in range(0, w, step):

            end_x = min(start_x + chunk_width, w)

            if end_x - start_x < 50:
                continue

            chunk = img[:, start_x:end_x]
            cw = end_x - start_x
            ch = h

            new_lines = []

            for cls, x1, y1, x2, y2 in boxes:

                # no overlap
                if x2 <= start_x or x1 >= end_x:
                    continue

                # clip to chunk
                nx1 = max(x1, start_x) - start_x
                nx2 = min(x2, end_x) - start_x
                ny1 = y1
                ny2 = y2

                # ignore too tiny clipped boxes
                if (nx2 - nx1) < 3 or (ny2 - ny1) < 3:
                    continue

                xc, yc, bw, bh = pixel_to_yolo(nx1, ny1, nx2, ny2, cw, ch)

                new_lines.append(
                    f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"
                )

            new_name = f"{stem}_chunk{chunk_id}"

            # save image
            cv2.imwrite(
                os.path.join(out_img_dir, new_name + ".jpg"),
                chunk
            )

            # save label
            with open(
                os.path.join(out_lbl_dir, new_name + ".txt"),
                "w"
            ) as f:
                f.write("\n".join(new_lines))

            chunk_id += 1

print("Done.")

sampleimages\DJI_0004_HS0600D\images\trainclasses
Image not found: classes
sampleimages\DJI_0004_HS0600D\images\trainframe_000000
sampleimages\DJI_0004_HS0600D\images\trainframe_002397
sampleimages\DJI_0004_HS0600D\images\trainframe_004794
sampleimages\DJI_0004_HS0600D\images\trainframe_007191
sampleimages\DJI_0004_HS0600D\images\trainframe_009588
sampleimages\DJI_0004_HS0600D\images\trainframe_011985
sampleimages\DJI_0004_HS0600D\images\valclasses
Image not found: classes
sampleimages\DJI_0004_HS0600D\images\valframe_014382
sampleimages\DJI_0004_HS0600D\images\valframe_016779
Done.
